# **1. Introduction**

This project involves working with the cleaned, normalized, and tokenized PerCQA dataset, which contains around 1,000 Persian-language questions and over 21,000 answers from the NiniSite Q\&A forum. The goal is to develop a semantic retrieval system that retrieves and ranks relevant answers based on a user’s query using advanced semantic similarity methods.

# **2. Environment Setup**

## 2.1 Install Requirements

In [3]:
! pip install transformers==4.48.2 FlagEmbedding lancedb --quiet

## 2.2 Import Libraries

In [74]:
import pandas as pd

from functools import cached_property
from pprint import pprint
from tabulate import tabulate
from FlagEmbedding import BGEM3FlagModel, FlagReranker
from lancedb import connect
from lancedb.embeddings.registry import register, get_registry
from lancedb.embeddings import TextEmbeddingFunction
from lancedb.pydantic import LanceModel, Vector
from lancedb.index import FTS

# **3. Loading Dataset**

In [5]:
df = pd.read_pickle('./data/ninisite_qa_clean.pkl')

In [6]:
df.head()

,QID,QCATEGORY,QDATE,QUSERID,QTYPE,QGOLD_YN,QUsername,QBody,QSubject,Comments,QBody_tokens,QSubject_tokens
0,1550088,None,2018-01-01T12:14:00,9101,General,Not Applicable,sami1366,دوستان نامزدم (شوهرم)باهام قهره امروز تولدشه چ...,قهر شوهرم,"[{'CID': '50758322', 'CUSERID': 4918, 'CGOLD':...","[دوس, نامزد, شوهر, با, قهره, امروز, تولدشه, ؟,...","[قهر, شوهر]"
1,1558609,None,2018-01-06T00:17:00,6856,General,Not Applicable,khanoomi72,من خودم حس میکنم اشتباهم گیر دادن بیخودی و غر ...,بزرگترین اشتباه دوران نامزدیتون چی بوده؟؟,"[{'CID': '51114707', 'CUSERID': 3935, 'CGOLD':...","[خود, حس, میکن, اشتباه, گیر, بیخود, غر, زدن, ز...","[بزرگ, اشتباه, دور, نامزدیتون, چ, ؟؟]"
2,1587820,None,2018-01-22T13:23:00,9918,General,Not Applicable,lol,سالگرد ازدواجم نزدیکه چی بخرم. سالگرد دوممونه....,کمک سالگرد ازدواجم نزدیکه,"[{'CID': '52336355', 'CUSERID': 10255, 'CGOLD'...","[سالگرد, ازدواج, نزدیکه, چ, بخر, سالگرد, دوممو...","[کمک, سالگرد, ازدواج, نزدیکه]"
3,4414921,None,2020-05-23T02:06:00,81403,General,Not Applicable,sepide1400,دختر من چهارده ماهه است. آیا برای تشخیص پرانتز...,متخصص ارتوپدی برای درمان پرانتزی پا,"[{'CID': '138006316', 'CUSERID': 38437, 'CGOLD...","[دخ, چهارده, ماهه, تشخیص, پرانتز, پا, سالگیا, ...","[متخصص, ارتوپد, در, پرانتز, پا]"
4,3109668,None,2019-08-13T12:07:00,25700,General,Not Applicable,نسا۷۹,چه پمادی بزنم و از چیه اینجور شده . واضح نیس...,لپ پسرم قرمز دون دونه,"[{'CID': '100859930', 'CUSERID': 33391, 'CGOLD...","[پماد, بزن, چیه, اینجور, واضح, نیس, بزور, از, ...","[لپ, پسر, قرمز, دون, دونه]"


# **4. Semantich Retrieval System**

In this section, we use bge-m3, a multilingual embedding model
capable of capturing semantic similarity across various languages. To store and search embeddings efficiently, also we use LanceDB, a vector database that is easy to setup and integrates well with modern embedding workflows.

## 4.1 BGE-M3 embedding model

`bge-m3` is a **multi-function embedding model** developed by **BAAI** (Beijing Academy of Artificial Intelligence) as part of the **BGE (BAAI General Embedding)** family. It's designed for **universal embedding tasks** across multiple use cases:

- **Reranking**
- **Dense retrieval**
- **Classification**
- **Clustering**
- **Semantic similarity**

In [9]:
bge_m3_model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

sparse_linear.pt:   0%|          | 0.00/3.52k [00:00<?, ?B/s]

### 4.1.1 Sample Embedding

In [12]:
sample_qa = df.sample(1).iloc[0]

In [13]:
sample_embeddings = bge_m3_model.encode([sample_qa['QBody']])

In [15]:
pprint(sample_embeddings, width=30, indent=2, compact=False)

{ 'colbert_vecs': None,
  'dense_vecs': array([[-0.03934 , -0.03955 , -0.04538 , ...,  0.005116, -0.01845 ,
         0.01352 ]], dtype=float16),
  'lexical_weights': None}


In [16]:
sample_embeddings['dense_vecs'].shape

(1, 1024)


Explanation of Each Component

1. **`dense_vecs`**

  * **Type:** NumPy array of shape `(1, D)` where `D` is the embedding dimension (e.g., 768 or 1024).
  * **Meaning:** This is the **dense vector embedding** of your input text (`QBody` here). It is a fixed-length continuous vector that captures the semantic meaning of the text in a dense numerical space.
  * **Use cases:**

  * Semantic search (compute cosine similarity or dot product between query and document embeddings).
  * Text clustering or classification.
  * Any downstream NLP task that benefits from vector representations of text.


2. **`colbert_vecs`**

  * **Type:** Often a 3D tensor or list of vectors per token, but here it is `None` (disabled).
  * **Meaning:**

    * If enabled, these are **token-level embeddings** used in **ColBERT-style late interaction retrieval**.
    * Instead of a single vector per text, you get a set of vectors—one per token—enabling fine-grained token-level similarity computations.
  * **Use cases:**

    * Advanced retrieval methods that compare token-to-token similarity between query and document instead of global vectors.
    * Improves precision in ranking by modeling fine-grained matches..

3. **`lexical_weights`**

  * **Type:** Could be a vector or matrix, but here is `None`.
  * **Meaning:**

    * These typically represent **lexical importance or sparse weights** derived from token-level information.
    * Used for sparse retrieval methods, similar to TF-IDF or BM25 style weights but learned by the model.
  * **Use cases:**

    * Combine dense embeddings with sparse lexical signals in **hybrid retrieval** systems to capture exact token matches as well as semantic similarity.


| Component         | What it is                                  | When to use                     |
| ----------------- | ------------------------------------------- | ------------------------------- |
| `dense_vecs`      | Fixed-length dense text embedding           | Semantic search, classification |
| `colbert_vecs`    | Token-level embeddings for late interaction | Fine-grained retrieval, ranking |
| `lexical_weights` | Sparse lexical importance weights           | Hybrid sparse + dense retrieval |


## 4.2 LanceDB

LanceDB is an open-source vector database designed for storing, indexing, and searching large-scale vector embeddings efficiently. It’s built to help developers build similarity search applications such as semantic search, recommendation systems, and AI-powered retrieval.

### 4.2.1 Custom Embedding Function

In [17]:
@register("bge-m3")
class BGEM3Embeddings(TextEmbeddingFunction):
  name: str = "BAAI/bge-m3"

  def __init__(self, **kwargs):
    super().__init__(**kwargs)
    self._ndims = None

  def generate_embeddings(self, texts):
    embeddings = self._embedding_model.encode(
        list(texts),
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False
    )
    return embeddings["dense_vecs"].tolist()

  def ndims(self):
    if self._ndims is None:
      self._ndims = len(self.generate_embeddings("foo")[0])
    return self._ndims

  @cached_property
  def _embedding_model(self):
    return BGEM3FlagModel(self.name, use_fp16=False)

### 4.2.2 Schema

In [18]:
embedding_fn = get_registry().get("bge-m3").create(use_fp16=False)

In [19]:
class NiniSiteQuestionSchema(LanceModel):
  qid: str
  qbody: str = embedding_fn.SourceField()
  embedding: Vector(embedding_fn.ndims()) = embedding_fn.VectorField()

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


### 4.2.3 Create Table

In [33]:
db = connect('./lanced_data')

In [38]:
tbl = db.create_table("ninsite_questions", schema=NiniSiteQuestionSchema)

In [39]:
records = (df[['QID', 'QBody']]
           .rename(columns={"QID": "qid", "QBody": "qbody"})
           .to_dict(orient="records"))

In [41]:
BATCH_SIZE = 100
total_records = len(records)

for i in range(0, total_records, BATCH_SIZE):
  batch = records[i:i + BATCH_SIZE]
  print(f"Inserting records {i + 1} to {min(i + BATCH_SIZE, total_records)} of {total_records}...")
  tbl.add(batch)

Inserting records 1 to 100 of 990...


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Inserting records 101 to 200 of 990...


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

...

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Inserting records 901 to 990 of 990...


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


## 4.3 Search

In [47]:
queries = [
    'کسی تجربه‌ای از مراجعه به متخصص کودکان تو منطقه اندیشه داره؟',
    'با کفش‌های کثیف بچه‌ها تو خونه چطور رفتار می‌کنید؟',
    'چطور با شریک زندگی‌مون کنار بیایم وقتی شرایط شغلی‌ش خوب پیش نمی‌ره؟',
    'اگه بچه از روی کانتر یا اپن افتاده باشه، چه علائمی خطرناک محسوب میشه؟',
    'تو مراسم بله‌برون تو شیراز چه چیزهایی می‌برن؟',
]

In [60]:
def print_search_result(results: pd.DataFrame):
  if results.empty:
    print("No results found")
    return

  display_df = pd.DataFrame({
      'QID': results['qid'],
      'Question': results['qbody']
  })

  print(tabulate(
      display_df,
      headers=['QID', 'Question'],
      tablefmt='fancy_grid',
      showindex=True,
      maxcolwidths=[20, 80]
  ))
  print()

### 4.3.1 Vectore Embedding Search

In [81]:
for query in queries:
  print(f"\n{'='*80}")
  print(f"QUERY: {query}")
  print(f"{'='*80}")

  query_embedding = bge_m3_model.encode([query])['dense_vecs'][0]
  results = tbl.search(query_embedding).limit(5).to_pandas()

  print_search_result(results)


QUERY: کسی تجربه‌ای از مراجعه به متخصص کودکان تو منطقه اندیشه داره؟
╒════╤═════════╤══════════════════════════════════════════════════════════════════════════════════════════════════════════════════╕
│    │     QID │ Question                                                                                                         │
╞════╪═════════╪══════════════════════════════════════════════════════════════════════════════════════════════════════════════════╡
│  0 │ 5523051 │ تلفن و آدرس متخصص اطفال خوب سراغ دارید توی شهریار بهم  بگید؟                                                     │
├────┼─────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│  1 │ 3385311 │ میگن اختلال درخواب اطفال رو درمان میکنه . من دیگه بریدم بااین مشکل پسرم. این دکتر مطبش کجاست؟ تهرانه؟ خوبه ببرم؟ │
├────┼─────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│  2 │ 

### 4.3.2 Full Text Search

In [62]:
tbl.create_fts_index("qbody", use_tantivy=False)

In [72]:
print(tbl.list_indices())

[Index(FTS, columns=["qbody"], name="qbody_idx")]


In [73]:
for query in queries:
  print(f"\n{'='*80}")
  print(f"QUERY: {query}")
  print(f"{'='*80}")

  results = tbl.search(query, query_type="fts").limit(5).to_pandas()

  print_search_result(results)


QUERY: کسی تجربه‌ای از مراجعه به متخصص کودکان تو منطقه اندیشه داره؟
╒════╤═════════╤════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╕
│    │     QID │ Question                                                                                                                                                                                                                                                                   │
╞════╪═════════╪════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╡
│  0 │ 2452313 │ یک داندانپزشک خوب کودکان در شیراز که بدون بیهوشی کار دند

### 4.3.3 Hybrid Search

Hybrid search—combining **vector embeddings** (dense retrieval) and **full-text search** (sparse retrieval)—can be more effective than using either method alone because it leverages the strengths of both approaches while mitigating their weaknesses. Here’s why:

**1. Complementary Strengths**
- **Vector Embeddings (Dense Retrieval):**
  - Excels at **semantic understanding**, capturing meaning beyond exact keywords.
  - Handles **synonyms, paraphrasing, and conceptual queries** well (e.g., "automobile" vs. "car").
  - Struggles with **rare terms, proper nouns, or exact matches** (e.g., "Python 3.12 release notes").
  
- **Full-Text Search (Sparse Retrieval):**
  - Strong at **precise keyword matching** (e.g., "2024 Tesla Model Y specs").
  - Handles **rare terms, IDs, and exact phrases** effectively.
  - Fails at **semantic understanding** (e.g., "best budget laptop" vs. "cheap notebooks").

By combining both, hybrid search retrieves **semantically relevant** results while ensuring **keyword precision**.

**2. Improved Recall & Precision**
- **Recall:** Vector search ensures relevant but non-keyword-matching documents aren’t missed.
- **Precision:** Full-text search filters out irrelevant semantic matches that don’t contain key terms.
  
Example:  
- Query: *"How to optimize PostgreSQL queries for large datasets?"*  
  - **Full-text alone** might miss documents saying "speed up Postgres queries on big data."  
  - **Vector alone** might retrieve generic SQL optimization guides.  
  - **Hybrid** ensures both keyword matches ("PostgreSQL") and semantic relevance ("optimize large datasets") are included.

**3. Handling Diverse Query Types**
- **Factual queries** (e.g., "Elon Musk age") → Full-text excels.  
- **Conceptual queries** (e.g., "tips for startup success") → Vector excels.  
- **Mixed queries** (e.g., "latest AI advancements in healthcare") → Hybrid balances both.

**4. Mitigating Weaknesses**
- **Vector embeddings** can suffer from **low relevance** for niche terms.  
- **Full-text search** can’t generalize across synonyms (e.g., "ML" vs. "machine learning").  
Hybrid search **re-ranks** results, ensuring the best of both worlds.

Hybrid search outperforms single-method approaches because it:

- Captures **semantic meaning** + **keyword precision**.  
- Boosts **recall** without sacrificing **relevance**.  
- Adapts to **diverse query types**.  


### 4.3.4 Evaluation Methods

Search systems are typically evaluated using several key metrics that measure different aspects of performance:

- **Precision and Recall**

>**Precision** measures the proportion of retrieved documents that are actually relevant. If a search returns 10 documents and 7 are relevant, precision = 7/10 = 0.7.
>
>**Recall** measures the proportion of all relevant documents that were successfully retrieved. If there are 20 relevant documents total in the collection and the search found 7 of them, recall = 7/20 = 0.35.
>
>These metrics often trade off against each other - returning more results >typically increases recall but may decrease precision.

- **Mean Average Precision (MAP)**

>MAP calculates the average precision across multiple queries and considers the ranking order. For each query, you calculate Average Precision (AP) by looking at precision at each relevant document's position.
>
>Example: Query returns documents in order [relevant, irrelevant, relevant, relevant, irrelevant]
>- Precision at position 1: 1/1 = 1.0
>- Precision at position 3: 2/3 = 0.67
>- Precision at position 4: 3/4 = 0.75
>- AP = (1.0 + 0.67 + 0.75) / 3 = 0.81

- **Normalized Discounted Cumulative Gain (NDCG)**

>NDCG accounts for both relevance and ranking position, with higher-ranked relevant documents contributing more to the score. It's particularly useful when relevance isn't binary (documents can be somewhat relevant, very relevant, etc.).
>
>**Formula:**
>
>  $$
>  DCG_p = \sum_{i=1}^p \frac{2^{rel_i} - 1}{\log_2(i + 1)}
>  $$
>
>  where $rel_i$ is the relevance score of the document at position $i$.
>
>  Then, normalize by the ideal DCG (IDCG) to get nDCG:
>
>  $$
>  nDCG_p = \frac{DCG_p}{IDCG_p}
>  $$
>
>If documents are rated 0-3 for relevance and a search returns [3, 1, 2, 0, 3]:
>- DCG considers both the relevance score and position discount
>- NDCG normalizes this by the ideal possible ranking
>- Higher NDCG (closer to 1.0) indicates better ranking quality

- **Mean Reciprocal Rank (MRR)**

>MRR measures how quickly users find their first relevant result. It's the average of reciprocal ranks of the first relevant document across queries.
>
>Example: First relevant document appears at position 2 → reciprocal rank = 1/2 = 0.5
>If across 3 queries, first relevant documents appear at positions 1, 3, and 2:
>MRR = (1/1 + 1/3 + 1/2) / 3 = 0.61

- **Click-Through Rate (CTR) and User Engagement**

>These measure actual user behavior rather than relevance judgments:
>- **CTR**: Percentage of search results that users click on
>- **Session success rate**: Percentage of search sessions where users complete their task
>- **Time to click**: How quickly users find and click relevant results

---

Example Evaluation Scenario

Consider evaluating a search system for an e-commerce site:

Query: "wireless headphones"
- System returns 10 products
- 6 are actually wireless headphones (relevant)
- 4 are wired headphones or unrelated items

**Precision**: 6/10 = 0.6
**Recall**: Depends on total wireless headphones in catalog (if 50 exist, recall = 6/50 = 0.12)
**MRR**: If first relevant result is at position 2, MRR component = 1/2 = 0.5

The choice of metrics depends on the use case. E-commerce might prioritize precision to avoid showing irrelevant products, while academic search might emphasize recall to ensure researchers don't miss important papers. NDCG works well when you can rate relevance on a scale, while MRR is ideal for scenarios where users typically need just one good result.

## 4.4 Reranker

A reranker is a model used after the initial retrieval step (e.g., semantic search or hybrid search) to re-score and reorder the results based on finer-grained relevance.

In [75]:
bge_reranker = FlagReranker("BAAI/bge-reranker-base", use_fp16=True)

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

In [86]:
for query in queries:
  print(f"\n{'='*80}")
  print(f"QUERY: {query}")
  print(f"{'='*80}")

  query_embedding = bge_m3_model.encode([query])['dense_vecs'][0]
  results = tbl.search(query_embedding).limit(10).to_pandas()

  print("\nTop 10 Results (Semantic Search):")
  print_search_result(results)

  pairs = [[query, row["qbody"]] for _, row in results.iterrows()]
  rerank_scores = bge_reranker.compute_score(pairs)


  reranked = results.copy()
  reranked["score"] = rerank_scores
  reranked = reranked.sort_values("score", ascending=False).reset_index(drop=True)

  print("\nTop 5 Results (After Reranking):")
  print_search_result(reranked.head(5))


QUERY: کسی تجربه‌ای از مراجعه به متخصص کودکان تو منطقه اندیشه داره؟

Top 10 Results (Semantic Search):
╒════╤═════════╤══════════════════════════════════════════════════════════════════════════════════════════════════════════════════╕
│    │     QID │ Question                                                                                                         │
╞════╪═════════╪══════════════════════════════════════════════════════════════════════════════════════════════════════════════════╡
│  0 │ 5523051 │ تلفن و آدرس متخصص اطفال خوب سراغ دارید توی شهریار بهم  بگید؟                                                     │
├────┼─────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│  1 │ 3385311 │ میگن اختلال درخواب اطفال رو درمان میکنه . من دیگه بریدم بااین مشکل پسرم. این دکتر مطبش کجاست؟ تهرانه؟ خوبه ببرم؟ │
├────┼─────────┼────────────────────────────────────────────────────────────────────────────────────────